# Part 5 — Validation

Three checks confirm the construction. One benchmarks the representation. Two test
properties the build guarantees. None reports a biological result.

In [1]:
import numpy as np
import pandas as pd

import annnet as an
from uc2_common import *  # noqa: F403

G = load()
E = edges_frame(G)

## Native hyperedges vs. clique expansion

A tool without hyperedges expands each hyperedge into n(n−1)/2 pairwise edges. The
expansion also drops the signed metabolic stoichiometry, which has no pairwise
encoding. The cell below reports both costs: the 69,888 edges expand to 329,375, a
factor of 4.7.

In [2]:
an.to_nx(G)  # materialises fine
clique = sum(n * (n - 1) // 2 for n in
             (len(set(s.get("head", [])) | set(s.get("tail", [])) | set(s.get("members", [])))
              for s in G.hyperedge_definitions.values()) if n >= 2)
native = G.global_count("edges")
hyper = len(G.hyperedge_definitions)
print(f"AnnNet native edges     : {native:,}  ({hyper:,} hyperedges)")
print(f"clique-expanded edges   : {native - hyper + clique:,}  ({(native - hyper + clique) / native:.2f}x, "
      f"and signed stoichiometry lost)")

AnnNet native edges     : 69,888  (17,427 hyperedges)
clique-expanded edges   : 329,375  (4.71x, and signed stoichiometry lost)


## Null-DoRothEA permutation, and construction invariants

The permutation test rewires the regulatory layer at random and re-runs the Q1
query, 50 times. A random layer should not recover the real (TF, complex) pairs.
Across the 50 permutations, the mean overlap with the 1,290 real pairs was 0.10.
The invariant check then confirms that every promised coupling edge exists: 367 TF
anchors and 5,096 translation edges.

In [4]:
subunits = complex_subunits(G, min_size=3)
reg = E[E.edge_kind == "regulatory"]
tf_targets = reg.assign(tf=reg.source.str.removeprefix("gene:"),
                        tgt=reg.target.str.removeprefix("gene:")).groupby("tf")["tgt"].apply(set)
real = {(tf, cid) for cid, genes in subunits.items() for tf, tgts in tf_targets.items()
        if len(genes & tgts) / len(genes) >= 0.5 and len(genes & tgts) >= 2}

tfs = sorted(tf_targets.index)
genes_all = sorted({g for s in subunits.values() for g in s}.union(*tf_targets.values))
overlaps = []
for k in range(50):
    rng = np.random.default_rng(SEED + k + 1)
    fake = pd.DataFrame({"tf": rng.choice(tfs, len(reg)), "t": rng.choice(genes_all, len(reg))}) \
        .groupby("tf")["t"].apply(set)
    fake_pairs = {(tf, cid) for cid, gs in subunits.items() for tf, ts in fake.items()
                  if len(gs & ts) / len(gs) >= 0.5 and len(gs & ts) >= 2}
    overlaps.append(len(real & fake_pairs))
print(f"real (TF, complex) pairs: {len(real):,} | mean overlap with 50 nulls: {np.mean(overlaps):.2f}")

tf_names = set(reg.source.str.removeprefix("gene:"))
anchors = {f"anchor:{tf}" for tf in tf_names} & set(G.edge_definitions)
trans = {f"trans:{g}" for g in reg.target.str.removeprefix('gene:')} & set(G.edge_definitions)
print(f"invariants OK: {len(anchors):,} TF anchors, {len(trans):,} translation edges present")